# 任意の LXCat 断面積から COMSOL 用の EEDF と電子移動度を作る

LXCat 形式の電子衝突断面積を読み込み、**平均電子エネルギーの範囲を指定して** DC の swarm 計算を行い、
COMSOL の Plasma インターフェースで使うファイルを書き出す汎用ノートブックです。

- 主ガスと、ラジカル・励起種などのモル分率を指定して混合気体を組み立てます。断面積ファイルは成分ごとに
  指定でき、複数の気体を含むファイルからは標的（反応式の左辺）で過程を選べます。
- 読み込んだ断面積を、表の範囲の外へ外挿します（定数、0、線形、両対数（べき乗則）、多項式、Bethe 型）。
- 各平均エネルギー ⟨ε⟩ について、その値になる E/N を求根で求めます。エネルギー格子の上限も ⟨ε⟩ に合わせて
  自動で決めるので、⟨ε⟩ が小さい点も細かい格子で計算します。

書き出すファイル（既定は `examples/output/lxcat_to_comsol/`）

| ファイル | 内容 | COMSOL での使い方 |
|---|---|---|
| `comsol_eedf.txt` | EEDF f(ε, ⟨ε⟩)（V^(-3/2)、∫ε^(1/2) f dε = 1） | 2引数の Interpolation 関数 → 電子エネルギー分布関数 |
| `comsol_mobility.txt` | 換算電子移動度 μN(⟨ε⟩)（1/(V·m·s)） | 1引数の Interpolation 関数 → 換算電子移動度 |
| `cross_sections_<成分>.txt` | 外挿した断面積（LXCat 形式） | Cross Section Import（EEDF と同じ断面積で速度係数を計算させる） |
| `swarm_summary.csv` | E/N、ドリフト速度、移動度、電離・付着周波数、各過程の速度係数 | 確認用 |

使い方: 1〜3 の設定のセルを書き換えて、上から順に実行します。既定の例（Ar と Ar*、⟨ε⟩ = 0.1〜20 eV の30点）は
数十秒で終わります。

In [ ]:
import csv
import dataclasses
import math
import time
import warnings
from collections import Counter
from datetime import date
from pathlib import Path

import numpy as np

import boltzpmp as bp

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("matplotlib がないので図は描きません（uv pip install matplotlib）")

# EEPF の裾は、このノートブックで格子の上限を調整して確かめるので、ソルバーの警告は出さない
warnings.filterwarnings("ignore", message="EEPF at eps_max.*")


def find_repo_root(start):
    """リポジトリ直下と examples のどちらから起動してもルートを見つける（見つからなければ None）。"""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "python" / "boltzpmp").is_dir():
            return candidate
    return None


ROOT = find_repo_root(Path.cwd().resolve()) or Path.cwd().resolve()
BUNDLED_DATA = Path(bp.__file__).parent / "data"  # boltzpmp に同梱の近似 Ar / Ar* 断面積
print(f"boltzpmp {bp.__version__}、作業フォルダ {ROOT}")

## 1. 断面積ファイルと混合気体の組成

`SPECIES` の1行が混合気体の1成分です。主ガスのほか、ラジカルや励起種を好きな数だけ並べられます。

- `name`: 混合気体での名前。ほかの過程の生成物（`Ar -> Ar*` の `Ar*`）と同じ名前にすると、その過程の逆過程
  （超弾性衝突）の標的がこの成分になります（`SUPERELASTIC = True` のとき）。
- `lxcat`: LXCat 形式のファイル。複数の成分で同じファイルを指定できます。
- `target`: ファイルから使う過程の標的（反応式の左辺。`Ar -> Ar*` なら `Ar`）。`None` ならファイルの全過程。
  複数の気体を含むファイル（O2 と O など）から1つの気体の過程を選ぶときに使います。文字列のリストも可。
- `fraction`: モル分率。`None` にした成分（1つだけ）は残り（1 − ほかの成分の和）になります。すべて数値で
  合計が1でなければ、合計で割って正規化します（モル比で指定できます）。
- `mass_amu`: 分子量。ELASTIC / EFFECTIVE の3行目に質量比がないときに使います。

既定の例は、LXCat の Ar（IST-Lisbon）を主ガスとし、準安定励起種 Ar* を 0.1% 混ぜたものです。Ar* の断面積は
boltzpmp 同梱の近似データで、4.2 eV からの段階電離だけを含みます。

In [ ]:
SPECIES = [
    dict(name="Ar", lxcat=ROOT / "examples" / "data" / "Ar_IST-Lisbon_LXCat.txt",
         target="Ar", fraction=None, mass_amu=39.948),
    dict(name="Ar*", lxcat=BUNDLED_DATA / "Ar_star.txt",
         target="Ar*", fraction=1.0e-3, mass_amu=39.948),
]

GAS_TEMPERATURE_K = 300.0        # 気体温度（弾性衝突による熱交換と、励起準位の占有に使う）
PRESSURE_PA = 133.0              # 数密度を決めるだけで、E/N で表した結果には影響しない
EXCITATION_TEMPERATURE_K = None  # 励起準位の占有の温度（None なら気体温度。BOLSIG+ の Excitation temperature）
SUPERELASTIC = True              # 超弾性衝突（逆過程）を入れる
GAS_HEATING = True               # 気体の熱運動による、弾性衝突のエネルギー交換を入れる
EFFECTIVE_TO_ELASTIC = True      # EFFECTIVE から同じ標的の非弾性断面積を引いて ELASTIC にする（BOLSIG+ と同じ）

## 2. 断面積の外挿

boltzpmp は、表の最初の点より下の断面積を0、最後の点より上を最後の値で一定として扱います。表が 100 eV や
1 keV で終わっていると、⟨ε⟩ の大きい条件では EEDF の裾が表の外に出るので、外挿の仕方で結果が変わります。
ここで選んだ方法で表を延ばしてからソルバーに渡し、延ばした表を LXCat 形式でも書き出します。

高エネルギー側（`high`）の方法。どれも表の最後の点で値が連続になるように合わせます。

| 方法 | 外挿 | パラメータ |
|---|---|---|
| `"none"` | 表のまま（boltzpmp では最後の値で一定） | |
| `"constant"` | 最後の値で一定 | |
| `"zero"` | 0 | |
| `"linear"` | 直線（傾きは表の端の点への最小二乗） | |
| `"loglog"` | 両対数で直線、つまりべき乗則 σ ∝ ε^p | `slope`（p を固定するとき） |
| `"polynomial"` | `degree` 次の多項式 | `degree`、`space`（`"log"`: 両対数、`"linear"`: 線形） |
| `"bethe"` | Bethe 型 σ = (a + b ln ε)/ε（電離や光学的に許容な励起の、高エネルギーでの漸近形） | |

低エネルギー側（`low`）は、表の最初の点より下を、しきい値（しきい値のない過程は 0 eV）まで延ばします。
`"none"`、`"constant"`、`"zero"`、`"linear"`、`"loglog"`、`"polynomial"` が使えます。

- あてはめに使う表の点は、既定では端からエネルギーが `fit_window` 倍（既定 3）の範囲です（高エネルギー側なら
  ε_last/3 ≤ ε ≤ ε_last）。`n_fit=5`（端から5点）や `fit_range_eV=(下限, 上限)` でも指定できます。
- `EXTRAPOLATION` のキーは `"default"`、衝突の種類（`"ELASTIC"` など）、個別の過程（`"成分名:過程名"`。
  4 のセルで一覧を表示します）で、後のものほど優先します（指定した項目だけを上書き）。
- 外挿した値が負になったところは0にします。`CAP_HIGH_ENERGY_GROWTH = True` なら、高エネルギー側の外挿は
  表の最後の値を超えません（あてはめで増加に転じたときの暴走を防ぐ）。

In [ ]:
EXTRAPOLATION = {
    "default": dict(high="loglog", low="none"),
    "ELASTIC": dict(low="constant"),
    "EFFECTIVE": dict(low="constant"),
    "IONIZATION": dict(high="bethe"),
    # 個別の過程の例:
    # "Ar:E + Ar -> E + Ar(1P1), Excitation": dict(high="linear", n_fit=3),
}
EXTRAPOLATE_UP_TO_EV = 1.0e5    # 高エネルギー側をこのエネルギーまで延ばす（それより上は最後の値で一定）
LOW_ENERGY_FLOOR_EV = 1.0e-4    # 低エネルギー側の loglog / polynomial はここまで（その下は一定）
POINTS_PER_DECADE = 40          # 延ばした部分の表の点の密度（ソルバーは点の間を線形補間する）
CAP_HIGH_ENERGY_GROWTH = True   # 高エネルギー側の外挿が表の最後の値を超えないようにする

## 3. 平均電子エネルギーの範囲と計算条件

各 ⟨ε⟩ について、その値になる E/N を求根で求めます（相対誤差 `MEAN_ENERGY_RTOL` まで）。気体の熱運動を入れる
とき（`GAS_HEATING = True`）は、⟨ε⟩ を熱平衡の値 3/2 kT_g（300 K で 0.039 eV）より大きくしてください。

エネルギー格子は一様で、刻みは ⟨ε⟩/`CELLS_PER_MEAN_ENERGY` です。上限 eps_max は ⟨ε⟩ に合わせて自動で決めます。
EEPF の末端（eps_max での値とピークの比）が `TAIL_MAX` より大きければ広げ、`TAIL_MIN` より小さければ狭めます
（セル数は eps_max/⟨ε⟩ に比例して変わります）。

In [ ]:
MEAN_ENERGY_RANGE_EV = (0.1, 20.0)  # (最小, 最大) [eV]
N_MEAN_ENERGIES = 30
MEAN_ENERGY_SPACING = "log"         # "log"（等比）か "linear"（等差）
MEAN_ENERGIES_EV = None             # 値の列を直接与えるとき（上の3つより優先）

MEAN_ENERGY_RTOL = 1e-4             # 求めた ⟨ε⟩ と目標値の相対差の許容値
EN_SEARCH_RANGE_TD = (1e-5, 1e5)    # E/N を探す範囲 [Td]
FIRST_GUESS_TD = 1.0                # 最初の ⟨ε⟩ で E/N を探し始める値 [Td]

CELLS_PER_MEAN_ENERGY = 30          # エネルギーの刻み = ⟨ε⟩ / この値
N_THETA = 32                        # 角度方向のセル数
SCHEME = "limiter"                  # 移流スキーム（"limiter" か "upwind"）
MAX_ITERATIONS = 1000               # 陰解法の反復回数の上限（収束しなければ、初期状態や格子を変えてやり直す）
EPS_MAX_OVER_MEAN = 10.0            # 格子の上限の初期値（⟨ε⟩ の何倍か。2点目からは前の点の比を使う）
TAIL_MAX = 1e-6                     # EEPF(eps_max)/ピーク がこれより大きければ上限を広げる
TAIL_MIN = 1e-12                    # これより小さければ上限を狭める（分解能のため）

OUTPUT_DIR = ROOT / "examples" / "output" / "lxcat_to_comsol"
N_EXPORT_POINTS = 2000              # COMSOL の EEDF の表のエネルギー点の数（すべての ⟨ε⟩ で共通）

## 4. 断面積の読み込み

各成分のファイルを読み、`target` で過程を選びます。一覧の `成分名:過程名` が、`EXTRAPOLATION` に個別の過程を
指定するときのキーです。

In [ ]:
def resolve_fractions(species):
    """`fraction=None` の成分を残りで埋める。すべて数値なら、合計で割って正規化する。"""
    given = [spec.get("fraction") for spec in species]
    if sum(value is None for value in given) > 1:
        raise ValueError("fraction=None（残り）にできる成分は1つだけです")
    known = [float(value) for value in given if value is not None]
    if any(not math.isfinite(value) or value < 0.0 for value in known):
        raise ValueError(f"モル分率は0以上の有限な値にしてください: {given}")
    if None in given:
        rest = 1.0 - sum(known)
        if rest < 0.0:
            raise ValueError(f"ほかの成分のモル分率の和が {sum(known):g} で、残りが負になります")
        return [rest if value is None else float(value) for value in given]
    total = sum(known)
    if total <= 0.0:
        raise ValueError("モル分率の合計が0です")
    if not math.isclose(total, 1.0, rel_tol=1e-9):
        print(f"モル分率の合計が {total:g} なので、合計で割って正規化します")
    return [value / total for value in known]


def load_species(spec):
    """LXCat ファイルを読み、`target` の過程を返す。"""
    path = Path(spec["lxcat"]).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"{spec['name']}: 断面積ファイルが見つかりません: {path}")
    sections = bp.parse_lxcat(path.resolve())
    if not sections:
        raise ValueError(
            f"{spec['name']}: {path.name} に電子衝突の過程がありません（イオンの断面積ではないか確かめてください）"
        )
    available = sorted({section.target for section in sections})
    target = spec.get("target")
    if target is None:
        if len(available) > 1:
            print(f"  注意: {path.name} には複数の標的 {available} があります。target で選べます")
        return path, sections
    wanted = {target} if isinstance(target, str) else set(target)
    sections = [section for section in sections if section.target in wanted]
    if not sections:
        raise ValueError(
            f"{spec['name']}: {path.name} に標的 {sorted(wanted)} の過程がありません（ファイルにある標的: {available}）"
        )
    return path, sections


NAMES = [spec["name"] for spec in SPECIES]
if len(set(NAMES)) != len(NAMES):
    raise ValueError(f"成分の名前が重複しています: {NAMES}")
FRACTIONS = dict(zip(NAMES, resolve_fractions(SPECIES)))
SOURCES = {}
ORIGINAL = {}
for spec in SPECIES:
    name = spec["name"]
    SOURCES[name], ORIGINAL[name] = load_species(spec)
    kinds = Counter(section.kind for section in ORIGINAL[name])
    print(f"{name}: モル分率 {FRACTIONS[name]:.6g}、{SOURCES[name].name} から {len(ORIGINAL[name])} 過程 {dict(kinds)}")
    if not kinds.keys() & {"ELASTIC", "EFFECTIVE"}:
        print("  ELASTIC / EFFECTIVE がないので、この成分は運動量移行に寄与しません")
if not any(section.kind in ("ELASTIC", "EFFECTIVE") for sections in ORIGINAL.values() for section in sections):
    raise ValueError("どの成分にも ELASTIC / EFFECTIVE の断面積がありません")

print()
for name, sections in ORIGINAL.items():
    for section in sections:
        energy = section.data[:, 0]
        print(
            f"  {name}:{section.name}  [{section.kind}、しきい値 {section.threshold:g} eV、"
            f"表 {energy[0]:g}〜{energy[-1]:g} eV]"
        )

## 5. 断面積の外挿と混合気体の組み立て

In [ ]:
HIGH_METHODS = ("none", "constant", "zero", "linear", "loglog", "polynomial", "bethe")
LOW_METHODS = ("none", "constant", "zero", "linear", "loglog", "polynomial")
RULE_DEFAULTS = dict(
    high="none", low="none", fit_window=3.0, n_fit=None, fit_range_eV=None, degree=2, space="log", slope=None
)


def rule_for(gas_name, section, kind):
    """"default"、衝突の種類、"成分名:過程名" の順に EXTRAPOLATION の設定を重ねる。"""
    rule = dict(RULE_DEFAULTS)
    for key in ("default", kind, f"{gas_name}:{section.name}"):
        rule.update(EXTRAPOLATION.get(key, {}))
    label = f"{gas_name}:{section.name}"
    unknown = set(rule) - set(RULE_DEFAULTS)
    if unknown:
        raise ValueError(f"{label}: 外挿の設定に不明な項目があります: {sorted(unknown)}")
    if rule["high"] not in HIGH_METHODS:
        raise ValueError(f"{label}: high={rule['high']!r} は {HIGH_METHODS} から選んでください")
    if rule["low"] not in LOW_METHODS:
        raise ValueError(f"{label}: low={rule['low']!r} は {LOW_METHODS} から選んでください")
    if rule["space"] not in ("log", "linear"):
        raise ValueError(f"{label}: space={rule['space']!r} は 'log' か 'linear' です")
    if int(rule["degree"]) < 1 or float(rule["fit_window"]) <= 1.0:
        raise ValueError(f"{label}: degree は1以上、fit_window は1より大きくしてください")
    if rule["n_fit"] is not None and int(rule["n_fit"]) < 2:
        raise ValueError(f"{label}: n_fit は2以上にしてください")
    return rule


def anchored_slope(dx, dy):
    """端の点を通る直線 dy = p dx の最小二乗の傾き p（使える点がなければ None）。"""
    denominator = float(np.dot(dx, dx))
    return float(np.dot(dx, dy)) / denominator if denominator > 0.0 else None


def fit_points(energy, values, side, rule):
    """あてはめに使う表の点（`fit_range_eV`、`n_fit`、`fit_window` の順に優先）。"""
    if rule["fit_range_eV"] is not None:
        low, high = rule["fit_range_eV"]
        keep = (energy >= low) & (energy <= high)
        return energy[keep], values[keep]
    n = rule["n_fit"]
    if n is None:
        ratio = float(rule["fit_window"])
        n = int(np.sum(energy >= energy[-1] / ratio) if side == "high" else np.sum(energy <= energy[0] * ratio))
        n = max(n, 3)  # 範囲に点が少なければ端から3点
    n = int(n)
    return (energy[-n:], values[-n:]) if side == "high" else (energy[:n], values[:n])


def extrapolated(energy, values, points, side, rule):
    """表 (energy, values) の外側の points での値（端の点で連続）と、方法の説明（ASCII）。

    負の値や増加の頭打ちはここではしない（呼び出し側で処理する）。
    """
    method = rule[side]
    e0, v0 = (energy[-1], values[-1]) if side == "high" else (energy[0], values[0])
    constant = np.full(points.shape, v0)
    if method == "zero":
        return np.zeros(points.shape), "zero"
    if method == "constant":
        return constant, "constant"
    x, y = fit_points(energy, values, side, rule)
    if method == "linear":
        slope = anchored_slope(x - e0, y - v0)
        if slope is None:
            return constant, "constant (no points to fit)"
        return v0 + slope * (points - e0), f"linear, slope {slope:.4g} m2/eV"
    if method == "polynomial" and rule["space"] == "linear":
        return polynomial(x, y, e0, v0, points, int(rule["degree"]), log=False)
    # ここから先は対数をとるので、端の値が0ならそのまま0とする
    if v0 <= 0.0:
        return np.zeros(points.shape), "zero (the table ends with 0)"
    if method == "bethe":
        use = x > 0.0
        b = anchored_slope(np.log(x[use] / e0), x[use] * y[use] - e0 * v0)
        if b is None:
            return constant, "constant (no points to fit)"
        return (e0 * v0 + b * np.log(points / e0)) / points, f"bethe (a + b ln E)/E, b = {b:.4g}"
    use = (x > 0.0) & (y > 0.0)
    if method == "loglog":
        p = rule["slope"]
        if p is None:
            p = anchored_slope(np.log(x[use] / e0), np.log(y[use] / v0))
        if p is None:
            return constant, "constant (no points to fit)"
        return v0 * (points / e0) ** float(p), f"power law E^{float(p):.4g}"
    return polynomial(x[use], y[use], e0, v0, points, int(rule["degree"]), log=True)


def polynomial(x, y, e0, v0, points, degree, log):
    """端の点を通る degree 次の多項式（log なら両対数で）を最小二乗であてはめて points で評価する。"""
    if log:
        x, y, points, e0, v0 = np.log(x), np.log(y), np.log(points), math.log(e0), math.log(v0)
    dx = x - e0
    keep = dx != 0.0
    degree = min(degree, int(keep.sum()))
    if degree == 0:
        values = np.full(points.shape, v0)
        return (np.exp(values) if log else values), "constant (no points to fit)"
    scale = float(np.max(np.abs(dx[keep])))
    basis = np.column_stack([(dx[keep] / scale) ** k for k in range(1, degree + 1)])
    coefficients = np.linalg.lstsq(basis, y[keep] - v0, rcond=None)[0]
    t = (points - e0) / scale
    values = v0 + sum(c * t**k for k, c in enumerate(coefficients, start=1))
    space = "log-log" if log else "linear"
    with np.errstate(over="ignore"):
        return (np.exp(values) if log else values), f"polynomial degree {degree} ({space})"


def extension_points(energy, side, method, floor):
    """表の外側に加える点。high は EXTRAPOLATE_UP_TO_EV まで、low は floor（しきい値か 0）まで。"""
    if method == "none" or (side == "low" and method == "zero"):
        return np.empty(0)
    if side == "high":
        last = float(energy[-1])
        top = EXTRAPOLATE_UP_TO_EV
        if last <= 0.0 or top <= last:
            return np.empty(0)
        if method == "zero":
            return np.array([last * (1.0 + 1e-6), top])
        if method == "constant":
            return np.array([top])
        n = max(1, math.ceil(math.log10(top / last) * POINTS_PER_DECADE))
        return np.geomspace(last, top, n + 1)[1:]
    first = float(energy[0])
    if first <= floor:
        return np.empty(0)
    if method in ("constant", "linear"):
        return np.array([floor])
    lowest = floor if floor > 0.0 else min(LOW_ENERGY_FLOOR_EV, 0.1 * first)
    n = max(1, math.ceil(math.log10(first / lowest) * POINTS_PER_DECADE))
    points = np.geomspace(lowest, first, n + 1)[:-1]
    return points if floor > 0.0 else np.concatenate(([0.0], points))


def extend_side(energy, columns, side, rule, floor):
    """表の片側を延ばす。(加える点, 各列の値, 説明) を返す。"""
    points = extension_points(energy, side, rule[side], floor)
    if points.size == 0:
        return points, [np.empty(0) for _ in columns], "none"
    anchor = energy[-1] if side == "high" else energy[0]
    # 0 eV の点（低エネルギー側の loglog / polynomial）は、その上の点の値で一定にする
    positive = points > 0.0 if rule[side] in ("loglog", "polynomial", "bethe") else np.ones(points.shape, bool)
    # 値が0を横切るところに点を加える（線形補間で表したとき、0 になる位置がずれないように）
    crossings = []
    for column in columns:
        grid = np.sort(np.append(points[positive], anchor))
        raw, _ = extrapolated(energy, column, grid, side, rule)
        for i in np.flatnonzero(raw[:-1] * raw[1:] < 0.0):
            a, b, fa = grid[i], grid[i + 1], raw[i]
            for _ in range(60):  # 二分法
                middle = 0.5 * (a + b)
                fm = extrapolated(energy, column, np.array([middle]), side, rule)[0][0]
                a, b, fa = (a, middle, fa) if fa * fm <= 0.0 else (middle, b, fm)
            crossings.append(0.5 * (a + b))
    points = np.unique(np.concatenate([points, crossings]))
    points = points[points != anchor]
    positive = points > 0.0 if rule[side] in ("loglog", "polynomial", "bethe") else np.ones(points.shape, bool)
    extended = []
    description = None
    for column in columns:
        values = np.empty(points.shape)
        raw, text = extrapolated(energy, column, points[positive], side, rule)
        values[positive] = raw
        if not positive.all():  # 0 eV の点
            values[~positive] = raw[0]
        if not np.all(np.isfinite(values)):
            raise ValueError(f"{side} 側の外挿が発散しました（{text}）。方法、degree、あてはめの範囲を見直してください")
        values = np.maximum(values, 0.0)
        if side == "high" and CAP_HIGH_ENERGY_GROWTH and np.any(values > column[-1]):
            values = np.minimum(values, column[-1])
            text += ", capped at the last value"
        extended.append(values)
        description = description or text
    return points, extended, description


def extrapolate_section(gas_name, section, kind):
    """1つの過程の表を外挿した CrossSection と、その要約を返す（kind は設定を選ぶときの衝突の種類）。"""
    rule = rule_for(gas_name, section, kind)
    energy = section.data[:, 0]
    columns = [section.data[:, 1]]
    if section.mt_data is not None:
        if not np.array_equal(section.mt_data[:, 0], energy):
            raise ValueError(f"{gas_name}:{section.name}: 運動量移行断面積の表のエネルギーが積分断面積と異なります")
        columns.append(section.mt_data[:, 1])
    floor = max(float(section.threshold), 0.0)
    low_points, low_values, low_text = extend_side(energy, columns, "low", rule, floor)
    high_points, high_values, high_text = extend_side(energy, columns, "high", rule, floor)
    new_energy = np.concatenate([low_points, energy, high_points])
    new_columns = [
        np.concatenate([low, column, high]) for low, column, high in zip(low_values, columns, high_values)
    ]
    notes = []
    if low_points.size:
        notes.append(f"extrapolated below {energy[0]:.6g} eV down to {new_energy[0]:.6g} eV: {low_text}")
    if high_points.size:
        notes.append(f"extrapolated above {energy[-1]:.6g} eV up to {new_energy[-1]:.6g} eV: {high_text}")
    comment = "\n".join(filter(None, [section.comment, *(f"boltzpmp lxcat_to_comsol: {note}" for note in notes)]))
    extended = dataclasses.replace(
        section,
        data=np.column_stack([new_energy, new_columns[0]]),
        mt_data=None if section.mt_data is None else np.column_stack([new_energy, new_columns[1]]),
        comment=comment,
    )
    summary = dict(
        key=f"{gas_name}:{section.name}", table=(energy[0], energy[-1]),
        low=low_text if low_points.size else "none", high=high_text if high_points.size else "none",
        end=columns[0][-1], top=(new_energy[-1], new_columns[0][-1]),
    )
    return extended, summary


def effective_to_elastic(gas_name, sections):
    """EFFECTIVE（全運動量移行断面積）から、同じ標的の EXCITATION・IONIZATION・ATTACHMENT を引いて ELASTIC にする。

    外挿する前の表で、EFFECTIVE の表の範囲だけで引く（その外は、変換した ELASTIC を外挿する）。
    """
    converted = []
    for section in sections:
        if section.kind != "EFFECTIVE":
            converted.append(section)
            continue
        inelastic = [
            other for other in sections
            if other.target == section.target
            and other.kind in ("EXCITATION", "IONIZATION", "ATTACHMENT")
            and other.threshold >= 0.0
        ]
        energy = np.unique(np.concatenate([
            section.data[:, 0],
            *(other.data[:, 0] for other in inelastic),
            [other.threshold for other in inelastic],
        ]))
        energy = energy[(energy >= section.data[0, 0]) & (energy <= section.data[-1, 0])]
        elastic = section.sigma(energy) - sum((other.sigma(energy) for other in inelastic), np.zeros(energy.shape))
        note = (
            f"converted from EFFECTIVE by subtracting {len(inelastic)} inelastic cross sections of {section.target}"
        )
        print(f"  {gas_name}:{section.name}: EFFECTIVE から {len(inelastic)} 個の非弾性断面積を引いて ELASTIC にしました")
        if np.any(elastic < 0.0):
            negative = energy[elastic < 0.0]
            print(f"    差が負になった {negative[0]:g}〜{negative[-1]:g} eV は0にしました")
        if section.mt_data is not None:
            print("    EFFECTIVE の表の3列目は使いません")
        converted.append(dataclasses.replace(
            section,
            kind="ELASTIC",
            data=np.column_stack([energy, np.maximum(elastic, 0.0)]),
            mt_data=None,
            comment="\n".join(filter(None, [section.comment, f"boltzpmp lxcat_to_comsol: {note}"])),
        ))
    return converted


PROCESSED = {}
SUMMARY = []
for name, sections in ORIGINAL.items():
    converted = effective_to_elastic(name, sections) if EFFECTIVE_TO_ELASTIC else sections
    PROCESSED[name] = []
    for original, section in zip(sections, converted):
        # EFFECTIVE から変換した ELASTIC にも、EXTRAPOLATION["EFFECTIVE"] の設定を使う
        extended, summary = extrapolate_section(name, section, original.kind)
        PROCESSED[name].append(extended)
        SUMMARY.append(summary)

KINDS = ("ELASTIC", "EFFECTIVE", "EXCITATION", "IONIZATION", "ATTACHMENT", "ROTATION")
for key in EXTRAPOLATION:
    if key not in {"default", *KINDS} and key not in {row["key"] for row in SUMMARY}:
        print(f"注意: EXTRAPOLATION のキー {key!r} に当てはまる過程がありません")

print("過程  [表の範囲]  低エネルギー側 / 高エネルギー側  σ(表の上端) → σ(延ばした上端)")
for row in SUMMARY:
    top_energy, top_value = row["top"]
    print(
        f"  {row['key']}  [{row['table'][0]:g}〜{row['table'][1]:g} eV]  {row['low']} / {row['high']}"
        f"  {row['end']:.3e} → {top_value:.3e} m² ({top_energy:g} eV)"
    )

GASES = [
    bp.Gas(spec["name"], FRACTIONS[spec["name"]], PROCESSED[spec["name"]], mass_amu=spec.get("mass_amu"))
    for spec in SPECIES
]
MIXTURE = bp.Mixture(GASES, p_Pa=PRESSURE_PA, T_K=GAS_TEMPERATURE_K, T_exc_K=EXCITATION_TEMPERATURE_K)
# 組み立てられるか（逆過程の標的、質量比など）を小さな格子で先に確かめる
_processes = bp.PMSolver(
    MIXTURE, eps_max_eV=10.0, d_eps_eV=0.5, n_theta=4, superelastic=SUPERELASTIC, gas_heating=GAS_HEATING
).processes()
_superelastic = sum(process["name"].endswith("(superelastic)") for process in _processes)
print(f"\n数密度 {MIXTURE.N:.4e} m^-3、衝突過程 {len(_processes)} 個（うち逆過程 {_superelastic} 個）")

In [ ]:
# 読み込んだ表（実線）と外挿した部分（破線）
if plt is not None:
    fig, axes = plt.subplots(1, len(PROCESSED), figsize=(7 * len(PROCESSED), 5), squeeze=False)
    for ax, (name, sections) in zip(axes[0], PROCESSED.items()):
        for index, (original, extended) in enumerate(zip(ORIGINAL[name], sections)):
            color = f"C{index % 10}"
            label = f"{original.kind}: {original.name}" if len(sections) <= 12 else None
            ax.loglog(original.data[:, 0], original.data[:, 1], color=color, label=label)
            if extended.kind != original.kind:  # EFFECTIVE から変換した ELASTIC
                ax.loglog(extended.data[:, 0], extended.data[:, 1], ":", color=color, label="ELASTIC (converted)")
                continue
            for part in (extended.data[:, 0] <= original.data[0, 0], extended.data[:, 0] >= original.data[-1, 0]):
                if part.sum() > 1:
                    ax.loglog(extended.data[part, 0], extended.data[part, 1], "--", color=color)
        peak = max(float(section.data[:, 1].max()) for section in sections)
        ax.set_ylim(peak * 1e-6, peak * 3)
        ax.set_xlim(1e-3, EXTRAPOLATE_UP_TO_EV)
        ax.set_xlabel("electron energy [eV]")
        ax.set_ylabel("cross section [m$^2$]")
        ax.set_title(f"{name} (x = {FRACTIONS[name]:.3g}): solid = table, dashed = extrapolated")
        ax.grid(True, which="both", alpha=0.25)
        if len(sections) <= 12:
            ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## 6. 平均電子エネルギーを指定した swarm 計算

⟨ε⟩ の小さい順に、次を繰り返します。

1. 格子の上限を eps_max = (前の点の eps_max/⟨ε⟩) × ⟨ε⟩ とする（最初の点は `EPS_MAX_OVER_MEAN` 倍）。
2. その格子で、平均エネルギーが ⟨ε⟩ になる E/N を探す。⟨ε⟩ は E/N とともに増えるので、それまでの点から
   ln(E/N) を外挿した値から始めて目標を挟む区間を探し、Brent 法で絞り込む。
3. EEPF の末端（eps_max での値とピークの比）が `TAIL_MAX` より大きければ eps_max を広げ、`TAIL_MIN` より
   小さければ狭めて、2 からやり直す。

陰解法が `MAX_ITERATIONS` 回で収束しないとき（低い E/N の Ar などで、格子によって起きることがあります）は、
同じ格子で解けた近くの E/N の解から始め直し、それでも収束しなければセル数を少し変えた格子でやり直します。

In [ ]:
K_B_EV = 8.617333262e-5  # ボルツマン定数 [eV/K]
THERMAL_MEAN_ENERGY_EV = 1.5 * K_B_EV * GAS_TEMPERATURE_K
CELL_RETRY_FACTORS = (0.8, 1.25, 0.65, 1.5)  # 収束しないとき、セル数にこの倍率を順に掛けてやり直す
CELL_LIMITS = (50, 5000)  # セル数の下限と上限
MAX_GRID_PASSES = 10
TAIL_GOAL = 1e-9  # 上限を決め直すときの、EEPF(eps_max)/ピーク の目標


class NotConverged(RuntimeError):
    """陰解法が MAX_ITERATIONS 回で収束しなかった。"""


def mean_energy_targets():
    if MEAN_ENERGIES_EV is not None:
        values = np.asarray(MEAN_ENERGIES_EV, dtype=float).ravel()
    else:
        low, high = MEAN_ENERGY_RANGE_EV
        if MEAN_ENERGY_SPACING == "log":
            values = np.geomspace(low, high, N_MEAN_ENERGIES)
        elif MEAN_ENERGY_SPACING == "linear":
            values = np.linspace(low, high, N_MEAN_ENERGIES)
        else:
            raise ValueError(f"MEAN_ENERGY_SPACING は 'log' か 'linear' です: {MEAN_ENERGY_SPACING!r}")
    if not np.all(np.isfinite(values)) or np.any(values <= 0.0):
        raise ValueError("平均エネルギーは正の有限な値にしてください")
    values = np.unique(values)
    if values.size < 2:
        raise ValueError("COMSOL の2引数の補間には、2点以上の平均エネルギーが必要です")
    if GAS_HEATING and values[0] <= 1.01 * THERMAL_MEAN_ENERGY_EV:
        raise ValueError(
            f"平均エネルギーは熱平衡の値 3/2 kT_g = {THERMAL_MEAN_ENERGY_EV:.4g} eV より大きくしてください"
            f"（最小が {values[0]:g} eV）"
        )
    return values


def make_solver(eps_max, n_cells):
    return bp.PMSolver(
        MIXTURE, eps_max_eV=eps_max, d_eps_eV=eps_max / n_cells, n_theta=N_THETA,
        superelastic=SUPERELASTIC, gas_heating=GAS_HEATING,
    )


def brent(f, a, b, fa, fb, ftol, xtol=1e-9, max_iterations=100):
    """fa と fb の符号が異なる区間 [a, b] で、f の根を Brent 法で探す。"""
    c, fc = a, fa
    d = e = b - a
    for _ in range(max_iterations):
        if fb * fc > 0.0:
            c, fc = a, fa
            d = e = b - a
        if abs(fc) < abs(fb):
            a, b, c = b, c, b
            fa, fb, fc = fb, fc, fb
        tol = 2.0 * np.finfo(float).eps * abs(b) + 0.5 * xtol
        half = 0.5 * (c - b)
        if abs(half) <= tol or abs(fb) <= ftol:
            return b
        if abs(e) >= tol and abs(fa) > abs(fb):
            s = fb / fa
            if a == c:
                p, q = 2.0 * half * s, 1.0 - s
            else:
                q, r = fa / fc, fb / fc
                p = s * (2.0 * half * q * (q - r) - (b - a) * (r - 1.0))
                q = (q - 1.0) * (r - 1.0) * (s - 1.0)
            if p > 0.0:
                q = -q
            p = abs(p)
            if 2.0 * p < min(3.0 * half * q - abs(tol * q), abs(e * q)):
                e, d = d, p / q
            else:
                d = e = half
        else:
            d = e = half
        a, fa = b, fb
        b += d if abs(d) > tol else math.copysign(tol, half)
        fb = f(b)
    return b


def steady_state(solver, x, solved):
    """E/N = exp(x) の定常解。

    既定の初期状態（二項近似の解）から収束しないとき（低い E/N の Ar などで起きる）は、この格子で解けた近くの
    E/N の解から始める。まだ解けた点がなければ、E/N を下げて解き（熱平衡に近いほど解きやすい）、そこから少しずつ
    E/N を上げる。
    """
    def solve(value, init_n=None):
        return solver.solve_dc(math.exp(value), scheme=SCHEME, max_steps=MAX_ITERATIONS, init_n=init_n)

    result = solve(x)
    if result.converged:
        return result
    for nearby in sorted(solved, key=lambda value: abs(value - x))[:2]:
        result = solve(x, solved[nearby].n)
        if result.converged:
            return result
    for k in range(1, 7):
        seed = solve(x - k * math.log(2.0))
        if not seed.converged:
            continue
        for value in np.linspace(x - k * math.log(2.0), x, 2 * k + 1)[1:]:
            result = solve(value, seed.n)
            if not result.converged:
                break
            seed = result
        else:
            return result
        break
    raise NotConverged(f"E/N = {math.exp(x):.4g} Td で {result.n_steps} 回反復しても収束しません")


def find_field(solver, target, guess, slope, trace):
    """格子を固定して、平均エネルギーが target になる ln(E/N) を探す。(ln E/N, 結果) を返す。

    guess は ln(E/N) の初期値、slope は d ln⟨ε⟩ / d ln(E/N) の見積もり。trace に (ln E/N, ln⟨ε⟩) を記録する。
    """
    lower, upper = (math.log(value) for value in EN_SEARCH_RANGE_TD)
    goal = math.log(target)
    ftol = math.log1p(MEAN_ENERGY_RTOL)
    solved = {}

    def f(x):
        result = steady_state(solver, x, solved)
        solved[x] = result
        trace.append((x, math.log(result.mean_energy)))
        return math.log(result.mean_energy) - goal

    # 目標を挟む区間を探す（⟨ε⟩ は E/N とともに増える）
    a = min(max(guess, lower), upper)
    fa = f(a)
    if abs(fa) <= ftol:
        return a, solved[a]
    direction = 1.0 if fa < 0.0 else -1.0
    step = min(max(abs(fa) / max(slope, 0.05), 0.02), 3.0)
    for _ in range(100):
        b = min(max(a + direction * step, lower), upper)
        if b == a:
            raise ValueError(
                f"⟨ε⟩ = {target:.4g} eV になる E/N が {EN_SEARCH_RANGE_TD} Td の範囲にありません"
                f"（{math.exp(a):.3g} Td で ⟨ε⟩ = {solved[a].mean_energy:.4g} eV）"
            )
        fb = f(b)
        if abs(fb) <= ftol:
            return b, solved[b]
        if fa * fb < 0.0:
            break
        # まだ同じ側: 割線で残りの幅を見積もって少し大きめに進む（ln で最大 3）
        remaining = fb * (b - a) / (fa - fb) if fa != fb else math.inf
        if remaining * direction > 0.0:
            step = min(max(1.5 * remaining * direction, 0.02), 3.0)
        else:
            step = min(2.0 * step, 3.0)
        a, fa = b, fb
    else:
        raise RuntimeError(f"⟨ε⟩ = {target:.4g} eV を挟む E/N の区間が見つかりません")
    x = brent(f, a, b, fa, fb, ftol)
    return x, solved[x]


def log_tail(result):
    """EEPF とそのピークの比の対数。"""
    eepf = np.maximum(np.asarray(result.eepf, dtype=float), 0.0)
    return np.log(np.maximum(eepf / eepf.max(), 1e-300))


def narrower_eps_max(result):
    """EEPF がピークの TAIL_GOAL まで下がるエネルギー（そのセルの上端）。"""
    above = np.flatnonzero(log_tail(result) > math.log(TAIL_GOAL))
    return float(result.mesh.eps_b[above[-1] + 1])


def wider_eps_max(result, eps_max):
    """EEPF の末端の傾きから、ピークの TAIL_GOAL まで下がるエネルギーを外挿する。"""
    ratio = log_tail(result)
    energy = np.asarray(result.energy_grid, dtype=float)
    k = max(3, energy.size // 10)
    slope = np.polyfit(energy[-k:], ratio[-k:], 1)[0]
    if slope < 0.0:
        estimate = energy[-1] + (math.log(TAIL_GOAL) - ratio[-1]) / slope
        return float(min(max(estimate, 1.3 * eps_max), 4.0 * eps_max))
    return 2.0 * eps_max


def solve_mean_energy(target, guess, slope, eps_ratio):
    """平均エネルギーが target になる swarm 解を、格子の上限を調整しながら求める。"""
    eps_max = eps_ratio * target
    factor = 1.0  # セル数の倍率（収束しないときに変える）
    retries = list(CELL_RETRY_FACTORS)
    trace = []
    for _ in range(MAX_GRID_PASSES):
        if eps_max > EXTRAPOLATE_UP_TO_EV:
            raise RuntimeError(
                f"⟨ε⟩ = {target:.4g} eV: EEDF の裾が断面積を外挿した範囲（{EXTRAPOLATE_UP_TO_EV:g} eV）を超えます"
            )
        n_cells = round(factor * CELLS_PER_MEAN_ENERGY * eps_max / target)
        n_cells = min(max(n_cells, CELL_LIMITS[0]), CELL_LIMITS[1])
        solver = make_solver(eps_max, n_cells)
        try:
            log_field, result = find_field(solver, target, guess, slope, trace)
        except NotConverged as error:
            if not retries:
                raise RuntimeError(
                    f"⟨ε⟩ = {target:.4g} eV: {error}。MAX_ITERATIONS、CELLS_PER_MEAN_ENERGY、N_THETA を"
                    "変えてみてください"
                ) from error
            factor = retries.pop(0)
            # これまでに計算した点のうち、目標に最も近い E/N から探し直す
            guess = min(trace, key=lambda point: abs(point[1] - math.log(target)))[0] if trace else guess
            continue
        guess = log_field
        tail = float(result.extra["eepf_tail_ratio"])
        if tail > TAIL_MAX:
            eps_max = wider_eps_max(result, eps_max)
            continue
        if tail < TAIL_MIN:
            narrower = narrower_eps_max(result)
            if narrower < 0.8 * eps_max:
                eps_max = narrower
                continue
        return dict(target=target, EN_Td=math.exp(log_field), result=result, eps_max=eps_max,
                    n_cells=n_cells, evaluations=len(trace))
    raise RuntimeError(f"⟨ε⟩ = {target:.4g} eV: 格子の上限が {MAX_GRID_PASSES} 回で決まりませんでした")


def predict_field(history, target):
    """それまでの解 (ln⟨ε⟩, ln E/N) から、ln(E/N) と d ln⟨ε⟩ / d ln(E/N) を見積もる。"""
    goal = math.log(target)
    if not history:
        return math.log(FIRST_GUESS_TD), 0.5
    if len(history) == 1:
        (y0, x0), = history
        return x0 + max(min((goal - y0) / 0.5, 4.0), -4.0), 0.5
    (y1, x1), (y2, x2) = history[-2:]
    slope = (y2 - y1) / (x2 - x1) if x2 != x1 else 0.5
    slope = min(max(slope, 0.02), 3.0)
    return x2 + max(min((goal - y2) / slope, 4.0), -4.0), slope

In [ ]:
# 断面積の表（外挿する前）の上端のうち、最も低いもの（表が0で終わる過程は除く）。これより上の電子の割合を表示する
DATA_LIMIT_EV = min(
    (
        float(section.data[-1, 0])
        for sections in ORIGINAL.values()
        for section in sections
        if section.data[-1, 1] > 0.0
    ),
    default=math.inf,
)
if SCHEME not in ("limiter", "upwind"):
    raise ValueError(f"SCHEME は 'limiter' か 'upwind' です（陰解法では blending を使えない）: {SCHEME!r}")
TARGETS = mean_energy_targets()
RECORDS = []
history = []
eps_ratio = EPS_MAX_OVER_MEAN
print(f"⟨ε⟩ = {TARGETS[0]:.4g}〜{TARGETS[-1]:.4g} eV の {TARGETS.size} 点（熱平衡の値 {THERMAL_MEAN_ENERGY_EV:.4g} eV）")
print(f"「表の外」は、断面積の表の上端のうち最も低い {DATA_LIMIT_EV:g} eV より上にある電子の割合（外挿の影響を受ける）")
print(" ⟨ε⟩ [eV]     E/N [Td]  v_d [m/s]   μN [1/(V m s)]  eps_max [eV]  セル  評価  表の外   時間 [s]")
started = time.perf_counter()
for target in TARGETS:
    t0 = time.perf_counter()
    guess, slope = predict_field(history, target)
    record = solve_mean_energy(target, guess, slope, eps_ratio)
    result = record["result"]
    RECORDS.append(record)
    history.append((math.log(result.mean_energy), math.log(record["EN_Td"])))
    eps_ratio = record["eps_max"] / target
    mobility = result.drift_velocity / (record["EN_Td"] * 1e-21)
    beyond = float(np.sum((result.eedf * result.mesh.d_eps)[result.energy_grid > DATA_LIMIT_EV]))
    print(
        f"{result.mean_energy:9.5g}  {record['EN_Td']:11.5g}  {result.drift_velocity:10.4g}  {mobility:14.5g}"
        f"  {record['eps_max']:12.4g}  {record['n_cells']:4d}  {record['evaluations']:4d}  {beyond:7.1e}"
        f"  {time.perf_counter() - t0:7.2f}"
    )
    if abs(result.mean_energy / target - 1.0) > 10.0 * MEAN_ENERGY_RTOL:
        print(f"  注意: 目標 {target:.5g} eV に届いていません（E/N に対して ⟨ε⟩ が不連続に変わっている可能性）")
print(f"合計 {time.perf_counter() - started:.1f} 秒")

MEAN_ENERGY = np.array([record["result"].mean_energy for record in RECORDS])
FIELD_TD = np.array([record["EN_Td"] for record in RECORDS])
DRIFT_VELOCITY = np.array([record["result"].drift_velocity for record in RECORDS])
REDUCED_MOBILITY = DRIFT_VELOCITY / (FIELD_TD * 1e-21)  # μN = v_d / (E/N) [1/(V m s)]
if np.any(np.diff(MEAN_ENERGY) <= 0.0):
    raise RuntimeError("求めた平均エネルギーが単調増加になっていません（目標の間隔を広げてください）")
print(f"目標の平均エネルギーとの相対差は最大 {np.max(np.abs(MEAN_ENERGY / TARGETS - 1.0)):.2e}")

## 7. 結果の確認

In [ ]:
if plt is not None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    shown = np.unique(np.linspace(0, len(RECORDS) - 1, min(len(RECORDS), 8)).round().astype(int))
    for index in shown:
        result = RECORDS[index]["result"]
        axes[0, 0].semilogy(result.energy_grid, result.eepf, label=f"{result.mean_energy:.3g} eV")
    axes[0, 0].set_xscale("log")
    axes[0, 0].set_ylim(1e-10 * max(RECORDS[i]["result"].eepf.max() for i in shown), None)
    axes[0, 0].set_xlabel("electron energy [eV]")
    axes[0, 0].set_ylabel("EEPF [eV$^{-3/2}$]")
    axes[0, 0].set_title("EEPF (legend: mean energy)")
    axes[0, 0].legend(fontsize=8)

    axes[0, 1].loglog(MEAN_ENERGY, FIELD_TD, "o-")
    axes[0, 1].set_xlabel("mean electron energy [eV]")
    axes[0, 1].set_ylabel("E/N [Td]")
    axes[0, 1].set_title("reduced electric field")

    axes[1, 0].loglog(MEAN_ENERGY, REDUCED_MOBILITY, "s-", color="tab:red")
    axes[1, 0].set_xlabel("mean electron energy [eV]")
    axes[1, 0].set_ylabel("reduced mobility $\\mu N$ [1/(V m s)]")
    axes[1, 0].set_title("reduced electron mobility")

    # 混合気体全体への寄与（速度係数 × 標的の割合）の大きい過程
    contribution = {}
    for record in RECORDS:
        result = record["result"]
        for key, value in result.rate_coefficients.items():
            contribution[key] = max(contribution.get(key, 0.0), value * result.fractions.get(key, 1.0))
    for key in sorted(contribution, key=contribution.get, reverse=True)[:8]:
        values = np.array([record["result"].rate_coefficients.get(key, 0.0) for record in RECORDS])
        if np.any(values > 0.0):
            axes[1, 1].loglog(MEAN_ENERGY[values > 0.0], values[values > 0.0], label=key)
    axes[1, 1].set_ylim(1e-22, None)
    axes[1, 1].set_xlabel("mean electron energy [eV]")
    axes[1, 1].set_ylabel("rate coefficient [m$^3$/s]")
    axes[1, 1].set_title("rate coefficients (largest contributions)")
    axes[1, 1].legend(fontsize=6)
    for ax in axes.flat:
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. COMSOL 用のファイルを書き出す

- EEDF: すべての ⟨ε⟩ で共通のエネルギー点（0 と、等比の `N_EXPORT_POINTS - 1` 点）に EEPF を補間し
  （対数で線形補間）、COMSOL の規格化 ∫ε^(1/2) f dε = 1 に合わせ直します。boltzpmp の `eedf`（eV⁻¹）
  ではなく `eepf`（eV^(-3/2)）が COMSOL の f(ε) に当たります。2列目は、書き出した EEDF から計算し直した
  平均エネルギー ∫ε^(3/2) f dε です（`export_comsol_eedf.py` と同じ）。
- 換算移動度: μN = v_d / (E/N)。引数はソルバーの平均エネルギーです。
- 断面積: ソルバーに渡したもの（外挿、EFFECTIVE の変換のあと）を、成分ごとに LXCat 形式で書き出し、
  読み直して表が一致することを確かめます。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EEDF_FILE = OUTPUT_DIR / "comsol_eedf.txt"
MOBILITY_FILE = OUTPUT_DIR / "comsol_mobility.txt"
SUMMARY_FILE = OUTPUT_DIR / "swarm_summary.csv"


def trapezoid(values, coordinates):
    return float(np.sum(0.5 * (values[1:] + values[:-1]) * np.diff(coordinates)))


def resample_eepf(result, grid):
    """セル中心の EEPF を grid へ補間し、∫ε^(1/2) f dε = 1 に規格化する。

    最初のセル中心より下は一定、最後のセル中心から格子の上端 eps_max までは最後の2セルの傾きで延ばし、その上は0。
    """
    energy = np.asarray(result.energy_grid, dtype=float)
    log_eepf = np.log(np.maximum(np.asarray(result.eepf, dtype=float), 1e-300))
    top = float(result.mesh.eps_b[-1])
    slope = min((log_eepf[-1] - log_eepf[-2]) / (energy[-1] - energy[-2]), 0.0)
    energy = np.append(energy, top)
    log_eepf = np.append(log_eepf, log_eepf[-1] + slope * (top - energy[-2]))
    values = np.exp(np.interp(grid, energy, log_eepf, left=log_eepf[0], right=-np.inf))
    values[values < 1e-280] = 0.0
    return values / trapezoid(np.sqrt(grid) * values, grid)


top = max(record["eps_max"] for record in RECORDS)
bottom = min(float(record["result"].energy_grid[0]) for record in RECORDS)
EXPORT_GRID = np.concatenate(([0.0], np.geomspace(bottom, top, N_EXPORT_POINTS - 1)))
EXPORT_EEPF = np.array([resample_eepf(record["result"], EXPORT_GRID) for record in RECORDS])
EXPORT_MEAN_ENERGY = np.array([trapezoid(EXPORT_GRID**1.5 * eepf, EXPORT_GRID) for eepf in EXPORT_EEPF])
if np.any(np.diff(EXPORT_MEAN_ENERGY) <= 0.0):
    raise RuntimeError("書き出す EEDF の平均エネルギーが単調増加になっていません（目標の間隔を広げてください）")
np.savetxt(
    EEDF_FILE,
    np.vstack([
        np.column_stack([EXPORT_GRID, np.full(EXPORT_GRID.shape, mean), eepf])
        for mean, eepf in zip(EXPORT_MEAN_ENERGY, EXPORT_EEPF)
    ]),
    fmt="%.10e", delimiter="\t", comments="% ", encoding="utf-8",
    header=(
        "COMSOL Spreadsheet data for a two-argument EEDF interpolation function\n"
        "Argument units in COMSOL: V, V; function unit: V^(-3/2)\n"
        "electron_energy_eV\tmean_electron_energy_eV\teedf_eV^-3/2"
    ),
)
print(f"{EEDF_FILE}: {len(RECORDS)} 本 × {EXPORT_GRID.size} 点（0〜{top:.4g} eV）")
print(
    "  2列目（書き出した EEDF の平均エネルギー）とソルバーの平均エネルギーの相対差は最大 "
    f"{np.max(np.abs(EXPORT_MEAN_ENERGY / MEAN_ENERGY - 1.0)):.1e}（ソルバーのエネルギー格子の離散化による）"
)

np.savetxt(
    MOBILITY_FILE, np.column_stack([MEAN_ENERGY, REDUCED_MOBILITY]), fmt="%.10e", delimiter="\t",
    comments="% ", encoding="utf-8",
    header=(
        "COMSOL Spreadsheet data for reduced electron mobility muN\n"
        "Argument unit in COMSOL: V; function unit: 1/(V*m*s)\n"
        "mean_electron_energy_eV\treduced_mobility_1_per_Vms"
    ),
)
print(f"{MOBILITY_FILE}: {len(RECORDS)} 点")


def safe_file_name(name):
    replaced = name.replace("*", "star").replace("+", "plus").replace("-", "minus")
    return "".join(character if character.isalnum() else "_" for character in replaced)


def lxcat_block(gas, section):
    """1つの過程を LXCat 形式の文字列にする。"""
    lines = [section.kind, section.species]
    if section.kind in ("ELASTIC", "EFFECTIVE"):
        lines.append(f" {gas.mass_ratio(section):.10e}")
    elif section.kind == "EXCITATION":
        weight = "" if section.weight_ratio is None else f" {section.weight_ratio:.10e}"
        lines.append(f" {section.threshold:.10e}{weight}")
    elif section.kind == "IONIZATION":
        lines.append(f" {section.threshold:.10e}")
    elif section.kind == "ROTATION":
        for energy, weight in (section.lower_state, section.upper_state):
            lines.append(f" {energy:.10e} {weight:.10e}")
    lines.append(f"SPECIES: e / {section.target}")
    lines.append(f"PROCESS: {section.name}")
    lines.extend(f"COMMENT: {line.strip()}" for line in section.comment.splitlines() if line.strip())
    table = section.data
    columns = "Energy (eV) | Cross section (m2)"
    if section.mt_data is not None:
        table = np.column_stack([section.data, section.mt_data[:, 1]])
        columns += " | Momentum transfer cross section (m2)"
    lines.append(f"COLUMNS: {columns}")
    lines.append("-----------------------------")
    lines.extend(" " + "\t".join(f"{value:.10e}" for value in row) for row in table)
    lines.append("-----------------------------")
    return "\n".join(lines)


def same_tables(written, reread):
    """書き出した過程と読み直した過程が一致するか。"""
    if [(s.kind, s.species, s.name) for s in written] != [(s.kind, s.species, s.name) for s in reread]:
        return False
    for section, again in zip(written, reread):
        tables = [(section.data, again.data), (section.mt_data, again.mt_data)]
        if (section.mt_data is None) != (again.mt_data is None) or not math.isclose(
            section.threshold, again.threshold, rel_tol=1e-9, abs_tol=1e-12
        ):
            return False
        for a, b in tables:
            if a is not None and (a.shape != b.shape or not np.allclose(a, b, rtol=1e-9, atol=0.0)):
                return False
    return True


CROSS_SECTION_FILES = {}
for gas in GASES:
    path = OUTPUT_DIR / f"cross_sections_{safe_file_name(gas.name)}.txt"
    header = [
        f"Electron-impact cross sections of {gas.name} used by boltzpmp {bp.__version__}"
        f" ({date.today().isoformat()}).",
        f"Source: {SOURCES[gas.name].name}",
        "Tables extended outside the original energy range are noted in the COMMENT lines.",
    ]
    blocks = [lxcat_block(gas, section) for section in gas.cross_sections]
    path.write_text("\n".join(header) + "\n\n" + "\n\n".join(blocks) + "\n", encoding="utf-8")
    CROSS_SECTION_FILES[gas.name] = path
    # 読み直して、ソルバーに渡した表と一致するか確かめる
    if not same_tables(gas.cross_sections, bp.parse_lxcat(path)):
        raise RuntimeError(f"{path}: 読み直した断面積が、ソルバーに渡したものと一致しません")
    print(f"{path}: {len(blocks)} 過程（読み直して一致を確認）")

PROCESS_KEYS = sorted({key for record in RECORDS for key in record["result"].rate_coefficients})
with SUMMARY_FILE.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow([
        "mean_energy_eV", "target_mean_energy_eV", "EN_Td", "drift_velocity_m_per_s",
        "reduced_mobility_1_per_V_m_s", "reduced_ionization_frequency_m3_per_s",
        "reduced_attachment_frequency_m3_per_s", "eps_max_eV", "n_cells", "iterations",
        *(f"k[{key}]_m3_per_s" for key in PROCESS_KEYS),
    ])
    for record, mobility in zip(RECORDS, REDUCED_MOBILITY):
        result = record["result"]
        writer.writerow([
            f"{result.mean_energy:.10e}", f"{record['target']:.10e}", f"{record['EN_Td']:.10e}",
            f"{result.drift_velocity:.10e}", f"{mobility:.10e}",
            f"{result.reduced_ionization_frequency:.10e}", f"{result.reduced_attachment_frequency:.10e}",
            f"{record['eps_max']:.6e}", record["n_cells"], result.n_steps,
            *(f"{result.rate_coefficients.get(key, 0.0):.10e}" for key in PROCESS_KEYS),
        ])
print(f"{SUMMARY_FILE}: {len(RECORDS)} 行、速度係数 {len(PROCESS_KEYS)} 過程（標的1個あたり）")

書き出した EEDF と断面積から、COMSOL と同じ式 k = γ ∫ε σ(ε) f(ε) dε（γ = (2e/m_e)^(1/2)）で速度係数を計算し、
boltzpmp の速度係数と比べます。比べるのは弾性衝突と、混合気体全体への寄与（速度係数 × 標的の割合）がその ⟨ε⟩ での
非弾性過程の最大値の 1e-3 以上の過程です。

差は、書き出した表の点の間隔と、ソルバーのエネルギー格子（boltzpmp はセル中心の断面積で速度係数を求める）から
生じます。数%を超えるときは `N_EXPORT_POINTS` や `CELLS_PER_MEAN_ENERGY` を増やしてください。

In [ ]:
GAMMA = math.sqrt(2.0 * 1.602176634e-19 / 9.1093837015e-31)  # (2e/m_e)^(1/2) [m/s/eV^(1/2)]
SECTIONS = {f"{gas.name}:{section.name}": section for gas in GASES for section in gas.cross_sections}
worst = (0.0, None, None)
for record, eepf in zip(RECORDS, EXPORT_EEPF):
    result = record["result"]
    weight = {key: result.rate_coefficients.get(key, 0.0) * result.fractions.get(key, 0.0) for key in SECTIONS}
    elastic = {key for key, section in SECTIONS.items() if section.kind in ("ELASTIC", "EFFECTIVE")}
    largest = max((value for key, value in weight.items() if key not in elastic), default=0.0)
    for key, section in SECTIONS.items():
        if result.rate_coefficients.get(key, 0.0) <= 0.0 or (key not in elastic and weight[key] < 1e-3 * largest):
            continue
        exported = GAMMA * trapezoid(EXPORT_GRID * section.sigma(EXPORT_GRID) * eepf, EXPORT_GRID)
        difference = exported / result.rate_coefficients[key] - 1.0
        if abs(difference) > abs(worst[0]):
            worst = (difference, result.mean_energy, key)
print(f"書き出した EEDF で計算した速度係数と boltzpmp の値の差は、最大 {worst[0]:+.2%}", end="")
print(f"（⟨ε⟩ = {worst[1]:.4g} eV、{worst[2]}）" if worst[2] else "")

## COMSOL での使い方

1. EEDF
   - Global Definitions → Functions → Interpolation を追加し、Data source: File、Data format: Spreadsheet、
     Number of arguments: 2 で `comsol_eedf.txt` を読み込みます。
   - 引数の単位を V, V、関数の単位を V^(-3/2) にします。第1引数が電子エネルギー、第2引数が平均電子エネルギーです。
   - Plasma インターフェースの電子エネルギー分布関数で Function を選び、この補間関数を指定します。
   - 表の ⟨ε⟩ の範囲の外は、Interpolation の外挿の設定（Constant など）で決まります。計算で現れる ⟨ε⟩ を
     覆うように `MEAN_ENERGY_RANGE_EV` を決めてください。
2. 換算電子移動度
   - もう1つ Interpolation を追加し、Number of arguments: 1 で `comsol_mobility.txt` を読み込みます。
     引数の単位は V、関数の単位は 1/(V\*m\*s) です。
   - 電子の輸送特性で換算電子移動度（Reduced electron mobility）を選び、この関数を平均電子エネルギーの
     関数として指定します。拡散係数、エネルギー移動度、エネルギー拡散係数は Einstein の関係から求めさせるのが
     標準です。
3. 断面積
   - Cross Section Import で `cross_sections_<成分>.txt` を読み込むと、EEDF と同じ（外挿した）断面積から
     速度係数が計算されます。元のファイルを読み込むと、表の外の扱いが boltzpmp の計算と異なることがあります。

## 注意

- 外挿した断面積は、データのない範囲についての仮定です。5 の図で外挿が妥当か確かめてください。⟨ε⟩ が大きい
  点ほど EEDF の裾が表の外に出て、外挿の影響を受けます（6 の表の「表の外」）。
- `swarm_summary.csv` の速度係数は、過程の標的1個あたりの値です（`rate_coefficients` と同じ）。
- 平均エネルギーの点の間隔が粗いと、COMSOL の補間で EEDF の形が不正確になります。EEDF の形が急に変わる範囲
  （Ar の Ramsauer 極小の付近など）では `N_MEAN_ENERGIES` を増やしてください。
- しきい値が ⟨ε⟩ よりずっと高い過程の速度係数は EEDF の裾だけで決まり、エネルギー格子の刻みに敏感です
  （He の三重項励起は ⟨ε⟩ = 3.75 eV で、刻み ⟨ε⟩/36 と格子を十分細かくしたときとで 9% 違いました）。
  そうした過程が重要なら `CELLS_PER_MEAN_ENERGY` を増やして、結果が変わらないか確かめてください。
- ただし、⟨ε⟩ が熱平衡の値に近い条件（E/N が 0.01 Td 以下など）では、格子を細かくすると結果がかえってずれる
  ことがあります。既定の例の Ar（⟨ε⟩ = 0.1 eV）では、`CELLS_PER_MEAN_ENERGY = 30` で二項近似の厳密解
  （Davydov 分布）との差が E/N で 0.3%、移動度で 2% でしたが、120 にすると移動度が 25% ずれました。